# **Uma DSL Híbrida para Jogos RPG com Suporte a Eventos de Gameplay**

Autores:
Gabriel Franklin Martins Lazzarini Miranda
Lucas Fortolan Sampaio

Resumo:
O desenvolvimento de jogos de Role-Playing Game (RPG) envolve a construção de narrativas interativas complexas, combinando a representação estrutural dos elementos da história com mecanismos dinâmicos de execução. No entanto, o uso de linguagens de programação de propósito geral para implementar esses sistemas frequentemente resulta em alto acoplamento entre a lógica de controle e a modelagem narrativa, dificultando a manutenção, a legibilidade e a evolução do código.

Para enfrentar esse desafio, este artigo apresenta o projeto, a implementação e a avaliação de uma Linguagem Específica de Domínio híbrida, ou Domain-Specific Language (DSL), voltada à modelagem de RPGs narrativos, culminando em sua versão final, V4. O sistema desenvolvido incorpora técnicas clássicas de construção de compiladores, incluindo análise léxica, sintática e semântica estática preventiva, baseada em uma gramática livre de contexto processada por algoritmos LALR(1), utilizando a biblioteca PLY (Python Lex-Yacc).

A execução do programa é realizada por meio de uma Árvore Sintática Abstrata, ou Abstract Syntax Tree (AST), interpretada em um ambiente Python. A versão V4 introduz avanços estruturais e lógicos significativos em relação às iterações anteriores: a integração de uma estrutura explícita de controle de fluxo condicional (if-then) apoiada por operadores relacionais, mutações granulares do estado das variáveis (set, add e remove), documentação explícita do código-fonte por meio de comentários e uma rotina ativa de prevenção de falhas em tempo de execução, governada por um laço de monitoramento condicional de Game Over.

Para garantir padrões elevados de engenharia de software, o ambiente de desenvolvimento foi totalmente refatorado, separando as células de modo a refletir o pipeline clássico dos compiladores tradicionais. A linguagem foi validada por meio da compilação e execução de um script experimental de grande escala intitulado “Horizonte Rubro e o Mar Invertido”.

Os resultados experimentais demonstraram que a DSL abstrai com sucesso as complexidades de programação de baixo nível em primitivas narrativas intuitivas e de alto nível, enquanto o analisador semântico estático preventivo elimina de forma eficaz falhas catastróficas em tempo de execução causadas por colisões de identificadores, incompatibilidades de tipo ou grafos de cenas órfãos, entregando uma arquitetura altamente desacoplada, segura e performática para narrativas interativas.

[GitHuB](https://https://github.com/AshesBorn/DSL-RPG)

Etapas:
* Dependências;
* Análise léxica;
* AST;
* Análise sintática;
* Análise semântica;
* Runtime;
* Script da DSL;
* Execução.

## 1. Instalação da biblioteca PLY

Use esta célula apenas quando estiver em um ambiente que ainda não tenha o PLY instalado, como uma sessão nova do Colab.


In [16]:
!pip install ply

## 2. Dependências

Todas as bibliotecas devem ser importadas no início.



In [17]:
# ============================================================
# DEPENDÊNCIAS DO PROJETO
# ============================================================
# Esta célula importa as bibliotecas necessárias para o interpretador.
# - os/sys: manipulação do ambiente e arquivos auxiliares.
# - time: efeito de impressão lenta no runtime.
# - ply.lex / ply.yacc: construção do lexer e parser.

import os
import time
import types
import sys
import ply.lex as lex
import ply.yacc as yacc

try:
    from IPython.display import clear_output
except ImportError:
    clear_output = None


## 3. Limpeza de cache do PLY

O PLY pode gerar arquivos auxiliares como `parsetab.py` e `parser.out`.
Remover esses arquivos evita conflitos quando a gramática é alterada.


In [18]:
# Objetivo: limpar arquivos temporários gerados pelo PLY.
# Isso evita que versões antigas da tabela sintática interfiram nos testes.

if os.path.exists("parsetab.py"):
  os.remove("parsetab.py")

if os.path.exists("parser.out"):
  os.remove("parser.out")


## 4. Análise léxica: palavras reservadas, tokens e expressões regulares

Nesta etapa, a linguagem define quais símbolos existem. O lexer reconhece palavras como `scene`, `event`, `choice`, operadores como `>=`, `==` e delimitadores como `{`, `}` e `;`.


In [19]:
# ============================================================
# CONFIGURAÇÃO LÉXICA: PALAVRAS RESERVADAS, TOKENS E REGEX
# ============================================================
# Esta célula define o vocabulário da DSL.
# O lexer usa essas definições para converter o código-fonte em tokens.

# ============================================================
# PALAVRAS RESERVADAS (SEMÂNTICA)
# ============================================================

reserved = {
  'scene': 'SCENE', # Declara uma cena do jogo
  'character': 'CHARACTER', # Declara um personagem e seus atributos
  'event': 'EVENT', # Declara um evento reutilizável

  'say': 'SAY', # Exibe uma fala/texto na tela
  'choice': 'CHOICE', # Cria opções para o jogador escolher
  'goto': 'GOTO', # Muda diretamente para outra cena
  'trigger': 'TRIGGER', # Dispara um evento declarado anteriormente

  # Utilizado para alterar os atributos
  'set': 'SET',
  'add': 'ADD',
  'remove': 'REMOVE',

  'if': 'IF',
  'then': 'THEN',

  'true': 'TRUE',
  'false': 'FALSE'
}

# ============================================================
# TOKENS (Linguagem Léxica)
# ============================================================

tokens = [
  'ID',
  'STRING',
  'NUMBER',

  'LBRACE', # Chave de abertura {
  'RBRACE', # Chave de fechamento }

  'SEMICOL', # Ponto e vírgula ;

  'EQUAL',

  'ARROW', # Seta de escolha ->

  'EQ', # Igualdade ==
  'NE', # Diferente !=
  'GT', # Maior que >
  'LT', # Menor que <
  'GE', # Maior ou igual >=
  'LE'  # Menor ou igual <=
] + list(reserved.values())

# ============================================================
# REGEX TOKENS
# ============================================================

t_LBRACE = r'\{'
t_RBRACE = r'\}'
t_SEMICOL = r';'

t_ARROW = r'->'

t_GE = r'>='
t_LE = r'<='
t_EQ = r'=='
t_NE = r'!='

t_GT = r'>'
t_LT = r'<'
t_EQUAL = r'='

t_ignore = ' \t'


## 5. Funções do lexer

Essas funções tratam tokens que precisam de processamento: strings perdem as aspas, números viram `int`, identificadores são comparados com palavras reservadas e quebras de linha atualizam o contador de linha.


In [20]:
# ============================================================
# FUNÇÕES DO LEXER
# ============================================================
# Cada função abaixo reconhece um tipo de token mais complexo.
# STRING remove as aspas; NUMBER converte texto em inteiro; ID diferencia identificadores de palavras reservadas.

# ============================================================
# STRING
# ============================================================

def t_STRING(t):
  r'"[^"]*"'
  t.value = t.value[1:-1]
  return t

# ============================================================
# NUMBER
# ============================================================

def t_NUMBER(t):
  r'\d+'
  t.value = int(t.value)
  return t

# ============================================================
# ID
# ============================================================

def t_ID(t):
  r'[a-zA-Z_][a-zA-Z0-9_]*'
  t.type = reserved.get(t.value, 'ID')
  return t


# ============================================================
# NEWLINE
# ============================================================

def t_newline(t):
  r'\n+'
  t.lexer.lineno += len(t.value)

# ============================================================
# ERROR
# ============================================================

def t_error(t):
  print(f'Erro léxico: {t.value[0]}')
  t.lexer.skip(1)




## 6. AST: nós da linguagem

A AST é a representação intermediária do programa. Em vez de executar o texto diretamente, o parser transforma a DSL em objetos como `SceneNode`, `ChoiceNode`, `IfNode` e `AddNode`. Isso deixa o projeto mais próximo de um interpretador real.


In [21]:
# ============================================================
# AST - ÁRVORE SINTÁTICA ABSTRATA
# ============================================================
# A AST representa o programa da DSL em objetos Python.
# O parser cria esses nós; o runtime interpreta esses nós depois.

# ============================================================
# AST
# ============================================================
class Node:
  pass

# ============================================================
# DECLARAÇÕES
# ============================================================

class SceneNode(Node):
  def __init__(self, name, commands):
    self.name = name
    self.commands = commands

class CharacterNode(Node):
  def __init__(self, name, attributes):
    self.name = name
    self.attributes = attributes

class EventNode(Node):
  def __init__(self, name, actions):
    self.name = name
    self.actions = actions

# ============================================================
# COMANDOS
# ============================================================

class SayNode(Node):
  def __init__(self, text):
    self.text = text

class GotoNode(Node):
  def __init__(self, target):
    self.target = target

class TriggerNode(Node):
  def __init__(self, event):
    self.event = event

class ChoiceNode(Node):
  def __init__(self, options):
    self.options = options

class SetNode(Node):
  def __init__(self, name, value):
    self.name = name
    self.value = value

class AddNode(Node):
  def __init__(self, name, value):
    self.name = name
    self.value = value

class RemoveNode(Node):
  def __init__(self, name, value):
    self.name = name
    self.value = value

# ============================================================
# CONDIÇÕES
# ============================================================

class IfNode(Node):
  def __init__(self, condition, command):
      self.condition = condition
      self.command = command

class ConditionNode(Node):
  def __init__(self, left, operator, right):
      self.left = left
      self.operator = operator
      self.right = right

# ============================================================
# ATRIBUTOS
# ============================================================

class AttributeNode(Node):
  def __init__(self, name, value):
      self.name = name
      self.value = value




## 7. Parser: regras gramaticais

O parser verifica se o código da DSL segue a estrutura esperada. Também cria os nós da AST. Por isso, esta célula precisa vir depois das classes da AST.


In [22]:
# ============================================================
# PARSER - GRAMÁTICA SINTÁTICA DA DSL
# ============================================================
# Esta célula define as regras gramaticais da linguagem usando PLY Yacc.
# Cada função p_* reconhece uma estrutura da DSL e cria nós da AST.

# PARSER
# ============================================================

def p_program(p):

  '''
  program : declarations
  '''

  p[0] = p[1]


# ============================================================
# DECLARATIONS
# ============================================================

def p_declarations_multiple(p):

  '''
  declarations : declarations declaration
  '''

  p[0] = p[1] + [p[2]]


def p_declarations_single(p):

  '''
  declarations : declaration
  '''

  p[0] = [p[1]]


# ============================================================
# SCENE
# ============================================================

def p_declaration_scene(p):

  '''
  declaration : SCENE ID LBRACE commands RBRACE
  '''

  p[0] = SceneNode(p[2], p[4])


# ============================================================
# CHARACTER
# ============================================================

def p_declaration_character(p):

  '''
  declaration : CHARACTER ID LBRACE attributes RBRACE
  '''

  p[0] = CharacterNode(p[2], p[4])


# ============================================================
# EVENT
# ============================================================

def p_declaration_event(p):

  '''
  declaration : EVENT ID LBRACE actions RBRACE
  '''

  p[0] = EventNode(p[2], p[4])


# ============================================================
# COMMANDS
# ============================================================

def p_commands_multiple(p):

  '''
  commands : commands command
  '''

  p[0] = p[1] + [p[2]]


def p_commands_single(p):

  '''
  commands : command
  '''

  p[0] = [p[1]]


# ============================================================
# SAY
# ============================================================

def p_command_say(p):

  '''
  command : SAY STRING SEMICOL
  '''

  p[0] = SayNode(p[2])


# ============================================================
# GOTO
# ============================================================

def p_command_goto(p):

  '''
  command : GOTO ID SEMICOL
  '''

  p[0] = GotoNode(p[2])


# ============================================================
# TRIGGER
# ============================================================

def p_command_trigger(p):

  '''
  command : TRIGGER ID SEMICOL
  '''

  p[0] = TriggerNode(p[2])

# ============================================================
# SET
# ============================================================

def p_command_set(p):

  '''
  command : SET ID EQUAL value SEMICOL
  '''

  p[0] = SetNode(p[2], p[4])


# ============================================================
# ADD
# ============================================================

def p_command_add(p):

  '''
  command : ADD ID value SEMICOL
  '''

  p[0] = AddNode(p[2], p[3])


# ============================================================
# REMOVE
# ============================================================

def p_command_remove(p):

  '''
  command : REMOVE ID value SEMICOL
  '''

  p[0] = RemoveNode(p[2], p[3])

# ============================================================
# CHOICE
# ============================================================

def p_command_choice(p):

  '''
  command : CHOICE LBRACE options RBRACE
  '''

  p[0] = ChoiceNode(p[3])


# ============================================================
# OPTIONS
# ============================================================

def p_options_multiple(p):

  '''
  options : options option
  '''

  p[0] = p[1] + [p[2]]


def p_options_single(p):

  '''
  options : option
  '''

  p[0] = [p[1]]


def p_option(p):

  '''
  option : STRING ARROW ID SEMICOL
  '''

  p[0] = (p[1], p[3])


# ============================================================
# ATTRIBUTES
# ============================================================

def p_attributes_multiple(p):

  '''
  attributes : attributes attribute
  '''

  p[0] = p[1] + [p[2]]


def p_attributes_single(p):

  '''
  attributes : attribute
  '''

  p[0] = [p[1]]


def p_attribute(p):

  '''
  attribute : ID EQUAL value SEMICOL
  '''

  p[0] = AttributeNode(p[1], p[3])


# ============================================================
# ACTIONS
# ============================================================

def p_actions_multiple(p):

  '''
  actions : actions action
  '''

  p[0] = p[1] + [p[2]]


def p_actions_single(p):

  '''
  actions : action
  '''

  p[0] = [p[1]]


# ============================================================
# IF
# ============================================================

def p_action_if(p):

  '''
  action : IF condition THEN command
  '''

  p[0] = IfNode(p[2], p[4])


# ============================================================
# CONDITION
# ============================================================

def p_condition(p):

  '''
  condition : ID operator value
  '''

  p[0] = ConditionNode(p[1], p[2], p[3])


# ============================================================
# OPERATORS
# ============================================================

def p_operator(p):

  '''
  operator : EQ
            | NE
            | GT
            | LT
            | GE
            | LE
  '''

  p[0] = p[1]


# ============================================================
# VALUES
# ============================================================

def p_value_number(p):

  '''
  value : NUMBER
  '''

  p[0] = p[1]


def p_value_string(p):

  '''
  value : STRING
  '''

  p[0] = p[1]


def p_value_true(p):

  '''
  value : TRUE
  '''

  p[0] = True


def p_value_false(p):

  '''
  value : FALSE
  '''

  p[0] = False


# ============================================================
# ERROR Sintático
# ============================================================

def p_error(p):
  if p:
    print("\n[ERRO SINTÁTICO]")
    print("Token:", p.type)
    print("Valor:", p.value)
    print("Linha:", p.lineno)

  else:
    print("\n[ERRO SINTÁTICO] EOF inesperado")


# ============================================================


## 8. Construção do lexer e parser

Só depois de definir tokens, funções léxicas e regras sintáticas é que o PLY pode construir os analisadores.


In [23]:
# ============================================================
# CONSTRUÇÃO DO LEXER E DO PARSER
# ============================================================
# Depois de declarar tokens e regras gramaticais, o PLY constrói os analisadores.

# PLY MODULE SETUP
# ============================================================
if '__main__' in sys.modules:
  main_module = sys.modules['__main__']
  if not hasattr(main_module, '__file__') or main_module.__file__ is None:
      main_module.__file__ = 'colab_notebook.py'
  if not hasattr(main_module, '__module__') or main_module.__module__ is None:
      main_module.__module__ = main_module.__name__

# ============================================================
# BUILD LEXER
# ============================================================

lexer = lex.lex(
  debug=False
)

# ============================================================
# BUILD PARSER
# ============================================================

parser = yacc.yacc(
  debug=False,
  write_tables=False,
  optimize=False
)


## 9. Análise semântica

A análise sintática verifica a forma do código. A análise semântica verifica o sentido: se cenas existem, se eventos foram declarados, se variáveis existem e se operações numéricas estão sendo aplicadas em variáveis numéricas.


In [24]:
# ============================================================
# ANÁLISE SEMÂNTICA
# ============================================================
# Esta etapa valida regras que a gramática sozinha não garante:
# - cena inicial existente;
# - variáveis declaradas antes do uso;
# - triggers apontando para eventos existentes;
# - choices/goto apontando para cenas existentes;
# - ADD/REMOVE apenas em variáveis numéricas;
# - SET sem troca indevida de tipo.
# ============================================================

class SemanticError(Exception):
    pass

def semantic_check(ast, start_scene="inicio"):
    if ast is None:
      raise SemanticError("AST inválida. O parser não gerou uma árvore sintática.")

    scenes = {}
    events = {}
    variables = {}

    errors = []

    # ========================================================
    # 1. COLETA DAS DECLARAÇÕES
    # ========================================================

    for node in ast:
        if isinstance(node, SceneNode):
          if node.name in scenes:
              errors.append(f"Cena duplicada: '{node.name}'.")
          scenes[node.name] = node

        elif isinstance(node, EventNode):
          if node.name in events:
              errors.append(f"Evento duplicado: '{node.name}'.")
          events[node.name] = node

        elif isinstance(node, CharacterNode):
          for attr in node.attributes:
              if attr.name in variables:
                  errors.append(f"Variável duplicada: '{attr.name}'.")
              variables[attr.name] = type(attr.value)

    # ========================================================
    # 2. VALIDAÇÕES GERAIS
    # ========================================================

    if len(scenes) == 0:
      errors.append("Nenhuma cena foi declarada.")

    if len(variables) == 0:
      errors.append("Nenhum personagem/atributo foi declarado.")

    if start_scene not in scenes:
      errors.append(f"Cena inicial '{start_scene}' não foi declarada.")

    # ========================================================
    # 3. FUNÇÕES AUXILIARES
    # ========================================================

    def is_int_type(value_type):
      # Verificação de tipo numérico se é int
      return value_type is int

    def is_int_value(value):
      return type(value) is int

    def check_variable_exists(var_name, context):
      if var_name not in variables:
          errors.append(f"Variável '{var_name}' não declarada em {context}.")
          return False
      return True

    def check_scene_exists(scene_name, context):
      if scene_name not in scenes:
          errors.append(f"Cena '{scene_name}' não declarada em {context}.")
          return False
      return True

    def check_event_exists(event_name, context):
      if event_name not in events:
          errors.append(f"Evento '{event_name}' não declarado em {context}.")
          return False
      return True

    def check_condition(condition, context):
      if not check_variable_exists(condition.left, context):
          return
      left_type = variables[condition.left]
      right_value = condition.right
      op = condition.operator

      # Comparações relacionais só fazem sentido com números
      if op in [">", "<", ">=", "<="]:
        if not is_int_type(left_type):
            errors.append(
                f"Condição inválida em {context}: variável '{condition.left}' precisa ser numérica para usar '{op}'."
            )
        if not is_int_value(right_value):
            errors.append(
                f"Condição inválida em {context}: valor comparado com '{op}' precisa ser numérico."
            )

      # Igualdade e diferença aceitam tipos iguais
      elif op in ["==", "!="]:
        if left_type is not type(right_value):
            errors.append(
                f"Condição inválida em {context}: comparação entre tipos diferentes em '{condition.left} {op} {right_value}'."
            )

    def check_command(command, context):
        # Recebe um comando da AST e verifica se ele está correto semanticamente.

        # SAY
        if isinstance(command, SayNode):
          return

        # GOTO
        elif isinstance(command, GotoNode):
          check_scene_exists(command.target, context)

        # TRIGGER
        elif isinstance(command, TriggerNode):
          check_event_exists(command.event, context)

        # CHOICE
        elif isinstance(command, ChoiceNode):
          if len(command.options) == 0:
              errors.append(f"Choice vazio em {context}.")
          for text, target in command.options:
              check_scene_exists(target, f"{context}, opção '{text}'")

        # SET
        elif isinstance(command, SetNode):
          if check_variable_exists(command.name, context):
              original_type = variables[command.name]
              new_type = type(command.value)
              if original_type is not new_type:
                  errors.append(
                      f"SET inválido em {context}: variável '{command.name}' não pode mudar de tipo."
                  )

        # ADD
        elif isinstance(command, AddNode):
          if check_variable_exists(command.name, context):
              if not is_int_type(variables[command.name]):
                  errors.append(
                      f"ADD inválido em {context}: variável '{command.name}' precisa ser numérica."
                  )
              if not is_int_value(command.value):
                  errors.append(
                      f"ADD inválido em {context}: valor somado em '{command.name}' precisa ser numérico."
                  )

        # REMOVE
        elif isinstance(command, RemoveNode):
          if check_variable_exists(command.name, context):
              if not is_int_type(variables[command.name]):
                  errors.append(
                      f"REMOVE inválido em {context}: variável '{command.name}' precisa ser numérica."
                  )
              if not is_int_value(command.value):
                  errors.append(
                      f"REMOVE inválido em {context}: valor subtraído em '{command.name}' precisa ser numérico."
                  )

        else:
          errors.append(f"Comando desconhecido em {context}: {type(command).__name__}")

    def check_action(action, context):
      if isinstance(action, IfNode):
          check_condition(action.condition, context)
          check_command(action.command, context)
      else:
          errors.append(f"Ação desconhecida em {context}: {type(action).__name__}")

    # ========================================================
    # 4. VALIDAÇÃO DAS CENAS
    # ========================================================
    for scene_name, scene in scenes.items():
      if len(scene.commands) == 0:
          errors.append(f"Cena '{scene_name}' não possui comandos.")
      for command in scene.commands:
          check_command(command, f"scene '{scene_name}'")

    # ========================================================
    # 5. VALIDAÇÃO DOS EVENTOS
    # ========================================================

    for event_name, event in events.items():
      if len(event.actions) == 0:
          errors.append(f"Evento '{event_name}' não possui ações.")
      for action in event.actions:
          check_action(action, f"event '{event_name}'")

    # ========================================================
    # 6. RESULTADO FINAL
    # ========================================================

    if errors:
      message = "\n[ERROS SEMÂNTICOS ENCONTRADOS]\n"
      for error in errors:
          message += f"- {error}\n"
      raise SemanticError(message)

    print("[ANÁLISE SEMÂNTICA] Nenhum erro encontrado.")
    print(f"Cenas declaradas: {len(scenes)}")
    print(f"Eventos declarados: {len(events)}")
    print(f"Variáveis declaradas: {len(variables)}")

## 10. Runtime / interpretador

O runtime percorre a AST e executa a lógica do jogo. Ele controla cena atual, status, inventário, eventos, escolhas, condições e término do jogo.


In [25]:
# ============================================================
# RUNTIME / INTERPRETADOR
# ============================================================
# Esta classe executa a AST gerada pelo parser.
# Ela controla as cenas, eventos, variáveis, status, inventário, escolhas e fim de jogo.

class Runtime:
    # Ambiente de execução da DSL.
    def __init__(self, ast):
      # Método construtor da classe.
      self.ast = ast
      self.scenes = {}
      self.events = {}
      self.variables = {}
      self.inventory = []
      self.running = True
      self.current_scene = None
      self.build_tables()

    # ========================================================
    # BUILD TABLES
    # ========================================================

    def build_tables(self):
      for node in self.ast:
          if isinstance(node, SceneNode):
              self.scenes[node.name] = node
          elif isinstance(node, EventNode):
              self.events[node.name] = node
          elif isinstance(node, CharacterNode):
              for attr in node.attributes:
                  self.variables[attr.name] = attr.value

    # ========================================================
    # PRINT
    # ========================================================

    def slow_print(self, text, speed=0.02):
      for char in text:
          print(char, end='', flush=True)
          time.sleep(speed)
      print()

    def clear_screen(self):
      try:
          clear_output(wait=True)
      except:
          os.system('cls' if os.name == 'nt' else 'clear')

    # ========================================================
    # STATUS
    # ========================================================

    def show_status(self):
      print("\n===================")
      print("STATUS")
      print("===================")

      itens_inventario = [
          "mapa",
          "suprimentos",
          "chave_mar",
          "fragmento_lenda",
          "fruta_estelar"
      ]

      for key, value in self.variables.items():
          if key not in itens_inventario:
              print(f"{key}: {value}")

      print("===================")

    # ========================================================
    # INVENTORY
    # ========================================================

    def show_inventory(self):
      print("\n===================")
      print("INVENTÁRIO")
      print("===================")

      itens_inventario = [
          "mapa",
          "suprimentos",
          "chave_mar",
          "fragmento_lenda",
          "fruta_estelar"
      ]

      nomes_itens = {
          "mapa": "Mapa",
          "suprimentos": "Suprimentos",
          "chave_mar": "Chave do Mar",
          "fragmento_lenda": "Fragmentos da Lenda",
          "fruta_estelar": "Fruta Estelar"
      }

      for item in itens_inventario:
          valor = self.variables.get(item, 0)
          print(f"{nomes_itens[item]}: {valor}")

      print("===================")

    # ========================================================
    # GAME OVER
    # ========================================================

    def check_game_over(self):
      if "vida" in self.variables and self.variables["vida"] <= 0:

          self.variables["vida"] = 0

          self.slow_print("\n===================")
          self.slow_print("GAME OVER")
          self.slow_print("===================")
          self.slow_print("Sua vida chegou a zero.")
          self.slow_print("A aventura terminou.")

          self.running = False

          return True

      if "energia" in self.variables and self.variables["energia"] <= 0:

          self.variables["energia"] = 0

          self.slow_print("\n===================")
          self.slow_print("GAME OVER")
          self.slow_print("===================")
          self.slow_print("Sua energia chegou a zero.")
          self.slow_print("Voce nao tem mais forcas para continuar.")

          self.running = False

          return True
      return False

    # ========================================================
    # RUN
    # ========================================================

    def run(self, start_scene):
      if start_scene not in self.scenes:
          raise Exception(f"Cena inicial '{start_scene}' não encontrada.")
      self.current_scene = self.scenes[start_scene]
      self.slow_print("=== RPG DSL ENGINE ===\n")
      self.show_status()
      if self.check_game_over():
          return
      while self.running:
          jumped = False

          for command in self.current_scene.commands:
              result = self.execute_command(command)
              if not self.running:
                  break
              if isinstance(result, str):
                  if result not in self.scenes:
                      raise Exception(f"Cena '{result}' não encontrada.")
                  self.current_scene = self.scenes[result]
                  jumped = True
                  break

          if not jumped:
              self.running = False

      self.slow_print("\n[FIM DO JOGO]")

    # ========================================================
    # EXECUTE COMMAND
    # ========================================================

    def execute_command(self, command):
      # SAY
      if isinstance(command, SayNode):
          self.slow_print(f"\n{command.text}")

      # GOTO
      elif isinstance(command, GotoNode):
          return command.target

      # TRIGGER
      elif isinstance(command, TriggerNode):
          self.slow_print(f"\n[EVENTO]: {command.event}")
          if command.event not in self.events:
              raise Exception(f"Evento '{command.event}' não declarado.")
          event = self.events[command.event]
          for action in event.actions:
              self.execute_action(action)
              if self.check_game_over():
                  return None

      # CHOICE
      elif isinstance(command, ChoiceNode):
          return self.handle_choice(command)

      # SET
      elif isinstance(command, SetNode):
          if command.name not in self.variables:
              raise Exception(f"Variável '{command.name}' não declarada.")
          valor_atual = self.variables[command.name]
          if type(valor_atual) is not type(command.value):
              raise Exception(
                  f"SET inválido: variável '{command.name}' não pode mudar de tipo."
              )
          self.variables[command.name] = command.value
          self.slow_print(f"\n[{command.name} definido como {command.value}]")
          self.check_game_over()

      # ADD
      elif isinstance(command, AddNode):
          if command.name not in self.variables:
              raise Exception(f"Variável '{command.name}' não declarada.")
          if type(self.variables[command.name]) is not int:
              raise Exception(
                  f"ADD só pode ser usado com variáveis numéricas. Variável inválida: {command.name}"
              )
          if type(command.value) is not int:
              raise Exception(
                  f"ADD só pode somar valores numéricos. Valor inválido em: {command.name}"
              )
          self.variables[command.name] += command.value
          self.slow_print(f"\n[{command.name} aumentou em {command.value}]")
          self.check_game_over()

      # REMOVE
      elif isinstance(command, RemoveNode):
          if command.name not in self.variables:
              raise Exception(f"Variável '{command.name}' não declarada.")
          if type(self.variables[command.name]) is not int:
              raise Exception(
                  f"REMOVE só pode ser usado com variáveis numéricas. Variável inválida: {command.name}"
              )
          if type(command.value) is not int:
              raise Exception(
                  f"REMOVE só pode subtrair valores numéricos. Valor inválido em: {command.name}"
              )
          self.variables[command.name] -= command.value

          if self.variables[command.name] < 0:
              self.variables[command.name] = 0

          self.slow_print(f"\n[{command.name} diminuiu em {command.value}]")

          self.check_game_over()

    # ========================================================
    # CHOICE
    # ========================================================

    def handle_choice(self, command):
      print("\n")
      self.slow_print("Escolha uma opção:\n")

      for i, option in enumerate(command.options):
          text, target = option
          print(f"[{i+1}] {text}")

      print("\n[S] Status")
      print("[I] Inventário")
      print("[Q] Sair")

      while True:
          value = input("\n> ").strip().lower()

          # STATUS
          if value == 's':

              self.show_status()
              continue

          # INVENTORY
          elif value == 'i':

              self.show_inventory()
              continue

          # QUIT
          elif value == 'q':

              self.running = False
              return None

          # OPÇÃO
          try:
              value = int(value)
              if 1 <= value <= len(command.options):
                _, target = command.options[value - 1]
                self.clear_screen()
                self.show_status()
                return target

          except:
              pass

          print("Escolha inválida.")

    # ========================================================
    # EVENTS
    # ========================================================

    def execute_action(self, action):
      if isinstance(action, IfNode):
          if self.evaluate_condition(action.condition):
              return self.execute_command(action.command)

    # ========================================================
    # CONDITIONS
    # ========================================================

    def evaluate_condition(self, condition):
      if condition.left not in self.variables:
          raise Exception(f"Variável '{condition.left}' não declarada.")

      left = self.variables[condition.left]
      right = condition.right
      op = condition.operator

      if op in ['>', '<', '>=', '<=']:
          if type(left) is not int or type(right) is not int:
              raise Exception(
                  f"Comparação inválida: operador '{op}' só pode ser usado com números."
              )

      if op == '==':
          return left == right

      elif op == '!=':
          return left != right

      elif op == '>':
          return left > right

      elif op == '<':
          return left < right

      elif op == '>=':
          return left >= right

      elif op == '<=':
          return left <= right

      return False


## 11. Código escrito na DSL

Esta célula contém o jogo em si. Ela deve ficar separada do interpretador para mostrar claramente a diferença entre:

- **linguagem implementada**: lexer, parser, AST, análise semântica e runtime;
- **programa escrito na linguagem**: a aventura do jogo.


In [26]:
# ============================================================
# SCRIPT ESCRITO NA DSL
# ============================================================
# A partir daqui não estamos mais definindo o interpretador.
# Estamos escrevendo um jogo usando a linguagem criada.
# Alterações na história devem ser feitas nesta célula.

code = '''

character hero {

    vida = 100;
    energia = 80;
    ouro = 50;
    coragem = 70;

    fama = 0;
    mapa = 0;
    suprimentos = 0;
    chave_mar = 0;
    fragmento_lenda = 0;
    fruta_estelar = 0;
}

event tempestade {

    if energia > 20 then say "Uma tempestade violenta atinge o navio. Ondas gigantes batem contra o casco.";
    if energia > 20 then add coragem 5;
    if energia > 20 then remove energia 20;
}

event comprar_suprimentos {

    if ouro >= 20 then say "Voce compra comida, cordas, remedios e ferramentas para a viagem.";
    if ouro >= 20 then add suprimentos 1;
    if ouro >= 20 then remove ouro 20;
}

event usar_suprimentos {

    if suprimentos > 0 then say "Voce usa os suprimentos da tripulacao e recupera parte das forcas.";
    if suprimentos > 0 then add vida 25;
    if suprimentos > 0 then remove suprimentos 1;
}

event batalha_corsarios {

    if vida > 30 then say "Corsarios mascarados invadem o conves. Espadas brilham, canhoes rugem e o navio treme.";
    if vida > 30 then say "Voce luta no meio da chuva, derruba dois inimigos e protege a tripulacao.";
    if vida > 30 then add coragem 10;
    if vida > 30 then add fama 10;
    if vida > 30 then remove vida 20;
}

event encontrar_mapa {

    if mapa == 0 then say "Dentro de uma garrafa antiga, voce encontra um mapa marcado com uma ilha que nao aparece em cartas comuns.";
    if mapa == 0 then say "No canto do papel esta escrito: Quando o mar subir ao ceu, a verdade afundada despertara.";
    if mapa == 0 then add coragem 5;
    if mapa == 0 then add mapa 1;
}

event ganhar_chave_mar {

    if chave_mar == 0 then say "Um velho navegador entrega a voce uma chave azulada feita de coral e metal.";
    if chave_mar == 0 then say "Ele diz: Esta chave nao abre portas. Ela abre caminhos no proprio mar.";
    if chave_mar == 0 then add chave_mar 1;
}

event despertar_fruta {

    if fruta_estelar == 0 then say "Voce encontra uma fruta brilhante, coberta por desenhos em espiral.";
    if fruta_estelar == 0 then say "Ao toca-la, uma energia estranha percorre seu corpo. O vento parece obedecer por um instante.";
    if fruta_estelar == 0 then add coragem 15;
    if fruta_estelar == 0 then add fruta_estelar 1;
}

event duelo_capitao {

    if vida > 40 then say "O Capitao Vargo, conhecido como Tubarao de Aco, bloqueia seu caminho.";
    if vida > 40 then say "Ele sorri e diz: Se quer a lenda do Mar Invertido, prove que merece navegar ate ela.";
    if vida > 40 then say "A luta e feroz. Lamina contra lamina. Vontade contra vontade.";
    if vida > 40 then add fama 25;
    if vida > 40 then add fragmento_lenda 1;
    if vida > 40 then remove vida 35;
}

event revelar_lenda {

    if fragmento_lenda > 0 then say "O fragmento revela parte da verdade: o maior tesouro nao e ouro, mas uma rota apagada do mundo.";
    if fragmento_lenda > 0 then say "Essa rota levaria ao primeiro oceano, onde todos os mares nasceram.";
}

event verificar_chave_mar {

    if chave_mar > 0 then say "A Chave do Mar brilha e abre uma passagem segura entre as ondas suspensas.";
}

scene inicio {

    say "Voce acorda no conves do navio Horizonte Rubro.";
    say "O mar esta calmo demais, o ceu esta vermelho e uma gaivota mecanica cruza as nuvens.";
    say "Seu capitao desapareceu durante a noite, deixando apenas uma frase gravada no leme:";
    say "Procurem o Mar Invertido antes que a Armada de Ferro o encontre.";

    choice {

        "Assumir o comando do navio" -> conves_navio;
        "Investigar a cabine do capitao" -> cabine_capitao;
        "Olhar o horizonte com a luneta" -> horizonte;
    }
}

scene conves_navio {

    say "A tripulacao espera uma decisao.";
    say "Alguns querem voltar ao porto. Outros querem seguir a ultima ordem do capitao.";
    say "No mar, bandeiras negras surgem entre a neblina.";

    choice {

        "Preparar o navio para batalha" -> ataque_corsarios;
        "Seguir para o Porto das Cem Velas" -> porto_cem_velas;
        "Investigar a cabine do capitao" -> cabine_capitao;
    }
}

scene cabine_capitao {

    say "A cabine esta revirada.";
    say "Mapas foram queimados, gavetas foram abertas e ha marcas de luta perto da janela.";
    say "Dentro de uma garrafa quebrada, algo ainda esta intacto.";

    trigger encontrar_mapa;

    choice {

        "Voltar ao conves" -> conves_navio;
        "Seguir a rota do mapa" -> mar_aberto;
        "Ir ao Porto das Cem Velas buscar informacoes" -> porto_cem_velas;
    }
}

scene horizonte {

    say "Pela luneta, voce ve uma ilha cercada por nuvens baixas.";
    say "Estranhamente, cachoeiras sobem do mar para o ceu.";
    say "A tripulacao chama esse lugar de Ilha do Mar Invertido.";

    add coragem 5;

    choice {

        "Seguir para a ilha misteriosa" -> mar_aberto;
        "Ir primeiro ao porto preparar a viagem" -> porto_cem_velas;
    }
}

scene porto_cem_velas {

    say "O Porto das Cem Velas e cheio de piratas, mercadores, cacadores de recompensa e navegadores mentirosos.";
    say "Nas paredes da taverna, cartazes prometem recompensa por qualquer um que procure o Mar Invertido.";
    say "A Armada de Ferro quer apagar essa lenda.";

    choice {

        "Comprar suprimentos por 20 de ouro" -> comprar;
        "Falar com o velho navegador" -> velho_navegador;
        "Entrar na taverna dos piratas" -> taverna_pirata;
        "Voltar ao navio" -> conves_navio;
    }
}

scene comprar {

    trigger comprar_suprimentos;

    choice {

        "Comprar mais suprimentos" -> comprar;
        "Falar com o velho navegador" -> velho_navegador;
        "Voltar ao navio" -> conves_navio;
    }
}

scene velho_navegador {

    say "O velho navegador observa seu mapa e fica em silencio.";
    say "Depois de alguns segundos, ele diz:";
    say "Esse mapa pertenceu aos primeiros piratas do mundo. Quem seguir essa rota encontrara a verdade que os reis enterraram.";

    trigger ganhar_chave_mar;

    choice {

        "Perguntar sobre o Mar Invertido" -> historia_mar_invertido;
        "Voltar ao porto" -> porto_cem_velas;
        "Partir imediatamente" -> mar_aberto;
    }
}

scene historia_mar_invertido {

    say "O velho explica que o Mar Invertido aparece apenas quando tres correntes se chocam.";
    say "Ali, o oceano sobe para o ceu e revela uma ilha escondida entre as nuvens.";
    say "Dizem que no centro da ilha existe uma fruta capaz de despertar a vontade do proprio mar.";

    add coragem 10;

    choice {

        "Partir para o Mar Invertido" -> mar_aberto;
        "Voltar ao porto" -> porto_cem_velas;
    }
}

scene taverna_pirata {

    say "A taverna esta cheia de risadas, brigas e canecas batendo nas mesas.";
    say "Um pirata ferido reconhece o simbolo do seu mapa.";
    say "Ele avisa que o Capitao Vargo tambem esta atras da ilha.";

    add fama 5;

    choice {

        "Desafiar os piratas da taverna" -> briga_taverna;
        "Sair antes de chamar atencao" -> porto_cem_velas;
    }
}

scene briga_taverna {

    say "A discussao vira pancadaria.";
    say "Mesas quebram, garrafas voam e alguem grita que voce tem espirito de capitao.";

    remove vida 10;
    add coragem 10;
    add fama 10;

    choice {

        "Voltar ao porto" -> porto_cem_velas;
        "Partir para o Mar Invertido" -> mar_aberto;
    }
}

scene ataque_corsarios {

    say "Os corsarios se aproximam rapidamente.";
    say "Eles nao querem ouro. Eles querem o mapa.";

    trigger batalha_corsarios;

    choice {

        "Perseguir o navio inimigo" -> navio_inimigo;
        "Fugir para o mar aberto" -> mar_aberto;
        "Usar suprimentos" -> curar_navio;
    }
}

scene curar_navio {

    trigger usar_suprimentos;

    choice {

        "Voltar para o conves" -> conves_navio;
        "Seguir para o mar aberto" -> mar_aberto;
    }
}

scene navio_inimigo {

    say "Voce salta para o navio inimigo no meio da fumaca dos canhoes.";
    say "No mastro principal, encontra uma bandeira da Armada de Ferro.";
    say "Eles estao cacando todos que conhecem a rota do Mar Invertido.";

    add fama 10;

    choice {

        "Roubar documentos da Armada" -> documentos_armada;
        "Voltar ao Horizonte Rubro" -> conves_navio;
    }
}

scene documentos_armada {

    say "Os documentos revelam que o governo dos mares teme uma antiga mensagem escondida na ilha.";
    say "Essa mensagem poderia unir piratas, povos livres e navegadores contra a Armada.";

    add fragmento_lenda 1;

    choice {

        "Seguir para o Mar Invertido" -> mar_aberto;
        "Voltar ao navio" -> conves_navio;
    }
}

scene mar_aberto {

    say "O Horizonte Rubro corta as ondas rumo ao desconhecido.";
    say "O vento muda de direcao varias vezes, como se o proprio oceano testasse sua decisao.";
    say "No terceiro dia, nuvens negras cercam o navio.";

    trigger tempestade;

    choice {

        "Manter a rota do mapa" -> corrente_invertida;
        "Voltar para o porto" -> porto_cem_velas;
        "Usar suprimentos" -> curar_mar;
    }
}

scene curar_mar {

    trigger usar_suprimentos;

    choice {

        "Manter a rota" -> corrente_invertida;
        "Voltar ao porto" -> porto_cem_velas;
    }
}

scene corrente_invertida {

    say "Tres correntes colidem ao redor do navio.";
    say "O mar comeca a subir como uma montanha liquida.";
    say "A tripulacao grita enquanto o Horizonte Rubro e levado para cima, navegando pelo ceu.";

    trigger verificar_chave_mar;

    choice {

        "Atravessar a corrente invertida" -> ilha_mar_invertido;
        "Tentar retornar antes que seja tarde" -> mar_aberto;
    }
}

scene ilha_mar_invertido {

    say "Voce chega a uma ilha impossivel.";
    say "Rios correm para cima, peixes nadam no ar e ruinas antigas flutuam sobre a praia.";
    say "No centro da ilha, uma torre de pedra guarda o segredo do primeiro oceano.";

    choice {

        "Explorar a praia suspensa" -> praia_suspensa;
        "Entrar na floresta de corais" -> floresta_corais;
        "Subir ate a torre antiga" -> torre_primeiro_oceano;
    }
}

scene praia_suspensa {

    say "Na praia, conchas gigantes repetem vozes do passado.";
    say "Uma delas sussurra o nome do seu capitao desaparecido.";
    say "Ele esteve aqui. E talvez ainda esteja vivo.";

    add coragem 10;

    choice {

        "Seguir as vozes" -> caverna_vozes;
        "Voltar para a ilha" -> ilha_mar_invertido;
    }
}

scene caverna_vozes {

    say "A caverna guarda ecos de antigos navegadores.";
    say "Nas paredes, desenhos mostram piratas enfrentando reis, monstros marinhos e tempestades vivas.";
    say "No fundo da caverna, voce encontra um fragmento da lenda.";

    add fragmento_lenda 1;

    choice {

        "Voltar para a praia" -> praia_suspensa;
        "Ir para a torre antiga" -> torre_primeiro_oceano;
    }
}

scene floresta_corais {

    say "A floresta de corais cresce fora da agua.";
    say "Entre os galhos vermelhos e azuis, uma fruta estranha pulsa como uma estrela viva.";

    trigger despertar_fruta;

    choice {

        "Comer a Fruta Estelar" -> poder_estelar;
        "Nao comer e seguir para a torre" -> torre_primeiro_oceano;
        "Voltar para a ilha" -> ilha_mar_invertido;
    }
}

scene poder_estelar {

    say "Ao comer a Fruta Estelar, seu corpo fica leve como vento.";
    say "Por alguns segundos, voce sente as correntes do mundo inteiro.";
    say "Mas tambem sente algo perigoso: a Armada de Ferro esta chegando.";

    add energia 30;
    add coragem 20;

    choice {

        "Correr para a torre antiga" -> torre_primeiro_oceano;
        "Voltar para a praia" -> praia_suspensa;
    }
}

scene torre_primeiro_oceano {

    say "A torre antiga se ergue no centro da ilha.";
    say "Cada degrau mostra uma batalha esquecida.";
    say "No topo, o Capitao Vargo espera com sua espada de aco negro.";

    trigger duelo_capitao;

    choice {

        "Enfrentar Vargo ate o fim" -> topo_torre;
        "Usar suprimentos antes da decisao final" -> curar_torre;
        "Recuar para a ilha" -> ilha_mar_invertido;
    }
}

scene curar_torre {

    trigger usar_suprimentos;

    choice {

        "Enfrentar Vargo" -> topo_torre;
        "Recuar para a ilha" -> ilha_mar_invertido;
    }
}

scene topo_torre {

    say "Depois do duelo, Vargo cai de joelhos e ri.";
    say "Ele diz: Agora entendo. Voce nao procura apenas tesouro. Voce procura liberdade.";
    say "A torre se abre, revelando uma mensagem gravada em pedra azul.";

    trigger revelar_lenda;

    choice {

        "Ler a mensagem antiga" -> mensagem_antiga;
        "Tomar o tesouro da torre" -> final_ganancia;
    }
}

scene mensagem_antiga {

    say "A mensagem revela que todos os mares ja foram um so.";
    say "Os reis antigos separaram os oceanos para dividir povos, controlar rotas e apagar a era dos navegadores livres.";
    say "O verdadeiro tesouro e a rota capaz de reunir os mares novamente.";
    say "Agora voce precisa decidir que tipo de capitao deseja ser.";

    choice {

        "Revelar a rota ao mundo" -> final_liberdade;
        "Guardar a rota apenas para sua tripulacao" -> final_pirata;
        "Destruir a mensagem para impedir uma guerra" -> final_sacrificio;
    }
}

scene final_liberdade {

    say "Voce retorna ao mar aberto e transmite a rota para todos os navios livres.";
    say "Piratas, pescadores, exploradores e povos esquecidos levantam suas velas.";
    say "A Armada de Ferro perde o controle das rotas.";
    say "Sua tripulacao nao encontrou apenas um tesouro. Encontrou o inicio de uma nova era.";

    add fama 100;
    add coragem 50;

    choice {

        "Ver status final" -> status_final;
    }
}

scene final_pirata {

    say "Voce guarda a rota apenas para sua tripulacao.";
    say "Com ela, o Horizonte Rubro cruza mares que ninguem mais consegue alcancar.";
    say "Voce se torna uma lenda viva, temido por reis e admirado por piratas.";
    say "Mas a liberdade do mundo tera que esperar.";

    add ouro 300;
    add fama 70;
    remove coragem 20;

    choice {

        "Ver status final" -> status_final;
    }
}

scene final_sacrificio {

    say "Voce destroi a mensagem antiga.";
    say "A rota desaparece, mas a Armada tambem jamais podera usa-la.";
    say "Sua tripulacao entende sua escolha em silencio.";
    say "Algumas verdades libertam. Outras podem incendiar o mundo antes da hora.";

    add coragem 80;
    remove energia 50;

    choice {

        "Ver status final" -> status_final;
    }
}

scene final_ganancia {

    say "Voce ignora a mensagem e toma os baus escondidos na torre.";
    say "O ouro e imenso, mas a ilha comeca a afundar no ceu.";
    say "O Mar Invertido rejeita aqueles que buscam apenas riqueza.";
    say "Voce escapa vivo, mas perde a chance de descobrir a maior verdade dos oceanos.";

    add ouro 500;
    remove coragem 50;
    remove fama 10;

    choice {

        "Ver status final" -> status_final;
    }
}

scene status_final {

    say "Sua jornada pelos mares chegou ao fim.";
    say "Use a opcao S para conferir seu status final antes de encerrar.";

    choice {

        "Finalizar aventura" -> fim;
    }
}

scene fim {

    say "Obrigado por jogar: Horizonte Rubro e o Mar Invertido.";
}

'''


## 12. Validação

Esta célula faz o parse e a análise semântica. Se houver erro, ele aparece antes do jogo começar. Isso facilita depuração.


In [27]:
# ============================================================
# VALIDAÇÃO DA DSL
# ============================================================
# Esta célula transforma o texto da DSL em AST e executa a análise semântica.
# Se houver erro de sintaxe ou semântica, ele aparece antes do jogo iniciar.

ast = parser.parse(code)
semantic_check(ast, start_scene="inicio")

print("DSL validada com sucesso. Pronta para execução.")


[ANÁLISE SEMÂNTICA] Nenhum erro encontrado.
Cenas declaradas: 32
Eventos declarados: 10
Variáveis declaradas: 10
DSL validada com sucesso. Pronta para execução.


## 13. Execução interativa

A execução fica separada porque usa `input()`. Assim você pode validar a DSL sem iniciar o jogo interativo toda vez.


In [28]:
# ============================================================
# EXECUÇÃO INTERATIVA
# ============================================================
# Execute esta célula apenas depois que a validação passar.
# O jogo usa input(), por isso deve ficar separado da célula de validação.

game = Runtime(ast)
game.run("inicio")



STATUS
vida: 65
energia: 90
ouro: 10
coragem: 125
fama: 25

Voce guarda a rota apenas para sua tripulacao.

Com ela, o Horizonte Rubro cruza mares que ninguem mais consegue alcancar.

Voce se torna uma lenda viva, temido por reis e admirado por piratas.

Mas a liberdade do mundo tera que esperar.

[ouro aumentou em 300]

[fama aumentou em 70]

[coragem diminuiu em 20]


Escolha uma opção:

[1] Ver status final

[S] Status
[I] Inventário
[Q] Sair

> S

STATUS
vida: 65
energia: 90
ouro: 310
coragem: 105
fama: 95

> Q

[FIM DO JOGO]
